# 3D Structure Viewer

In [1]:
!pip install py3dmol -q 

In [2]:
import py3Dmol
import numpy as np
import pandas as pd
from utils.pdb_parser import get_atom_coordinates_from_pdb, open_pdb
from utils.vector import calculate_direction, scalar_vector
from IPython.display import display, Markdown

In [12]:
# path
data_path_to_analyze = '../output/Test/2025-03-13T19.36.30-Sequence_Analyzer/Sequence_Analysis/'
pdbs_path = '../datasets/StarPep/ESMFold_pdbs/'

# Sequences with maximun distance

In [13]:
seqs_with_max_distance = pd.read_csv(data_path_to_analyze+'Sequences_With_Max_Distance.csv')
seqs_with_max_distance.head(7)

,distance_function,sequence,aa_src,aa_dst,length,esm2_perplexity,euclidean,canberra,lance_williams,clark,soergel,bhattacharyya,angular_separation
0,euclidean,EAIIRILQQLLFIHFRIGRRRRRRRR,0,25,26,1.739179,37.471738,2.880807,0.977003,1.666079,0.988368,7.132486,0.730023
1,canberra,IIRILQQLLFIHFRIGRRRRRRRR,0,1,24,1.643706,3.706152,2.999996,0.999999,1.732049,0.999999,2.442070,0.999999
2,lance_williams,EAIIRILQQLLFIHFRIGRRRRRRRR,0,3,26,1.739179,5.044255,2.999995,0.999999,1.732048,1.000000,2.686688,0.999998
3,clark,IIRILQQLLFIHFRIGRRRRRRRR,0,1,24,1.643706,3.706152,2.999996,0.999999,1.732049,0.999999,2.442070,0.999999
4,soergel,EAIIRILQQLLFIHFRIGRRRRRRRR,0,3,26,1.739179,5.044255,2.999995,0.999999,1.732048,1.000000,2.686688,0.999998
5,bhattacharyya,EAIIRILQQLLFIHFRIGRRRRRRRR,0,25,26,1.739179,37.471738,2.880807,0.977003,1.666079,0.988368,7.132486,0.730023
6,angular_separation,LQQLLFIHARIGRRRRRRRR,0,1,20,1.462829,3.698487,2.999996,0.999999,1.732048,0.999999,2.398538,0.999999


In [26]:
result = seqs_with_max_distance[seqs_with_max_distance['distance_function'] == 'euclidean']
int(result['aa_src'].iloc[0]) + 1

1

In [30]:
# functions
def get_data(seqs_with_max_distance, distance_function, dataset_path, pdbs_path):
    result = seqs_with_max_distance[seqs_with_max_distance['distance_function'] == distance_function]

    # Extract values from the result
    sequence = result['sequence'].iloc[0]
    euclidean_value = result['euclidean'].iloc[0]
    angular_separation_value = result['angular_separation'].iloc[0]
    residue_source = int(result['aa_src'].iloc[0]) + 1
    residue_target = int(result['aa_dst'].iloc[0]) + 1
    
    # Get PDB file based on the sequence
    sequences = pd.read_csv(dataset_path)
    sequence_id = sequences.loc[sequences['sequence'] == sequence, 'id'].iloc[0]
    pdb_file = f"{pdbs_path + sequence_id}.pdb"

    # Get atom coordinates
    pdb_data = open_pdb(pdb_file)
    coordinates = get_atom_coordinates_from_pdb(pdb_data, 'CA')
    residue_source_coord = coordinates[residue_source - 1]
    residue_target_coord = coordinates[residue_target - 1]

    # Calculate direction for both residues
    residue_source_dir = calculate_direction(residue_source_coord)
    residue_target_dir = calculate_direction(residue_target_coord)

    # Return the values in a dictionary
    data = {
        'sequence': sequence,
        'sequence_length': len(sequence),
        'euclidean_value': euclidean_value,
        'angular_separation_value': angular_separation_value,
        'residue_source': residue_source,
        'residue_target': residue_target,
        'pdb_file': pdb_file,
        'residue_source_coord': residue_source_coord,
        'residue_target_coord': residue_target_coord,
        'residue_source_dir': residue_source_dir,
        'residue_target_dir': residue_target_dir
    }

    return data

## 3D structure with Maximum Euclidean distance

In [34]:
data_eu = get_data(seqs_with_max_distance, 'euclidean', dataset_path, pdbs_path)

# display result
display(Markdown("<br>**Data with the maximum Euclidean distance:**<br>"))
display(Markdown(f"**Sequence:** {data_eu['sequence']}"))
display(Markdown(f"**Euclidean Distance:** {data_eu['euclidean_value']}"))
display(Markdown(f"**Angular Separation Distance:** {data_eu['angular_separation_value']}"))

# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_eu['pdb_file'], 'r').read(),'pdb')

view.setStyle({'cartoon': {'color':'spectrum', 'thickness':0.4, 'style':'rectangle'}})

view.rotate(90,'y',1);
view.zoom(0.055)

<br>**Data with the maximum Euclidean distance:**<br>

**Sequence:** EAIIRILQQLLFIHFRIGRRRRRRRR

**Euclidean Distance:** 37.47173836628632

**Angular Separation Distance:** 0.7300227506374359

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [33]:
# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_eu['pdb_file'], 'r').read(),'pdb')

view.addStyle({'and':[{'model':0}]}, {'cartoon': {'opacity':0.8,'color':'white'}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'stick': {'radius': 0.2, 'color':'grey', 'dashedBonds': False, 'singleBonds': False}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'sphere': {'radius': 0.3, 'color':'grey'}})
view.addStyle({'and':[{'model':0},{'atom':'CA'}]}, {'sphere': {'radius': 0.6, 'colorscheme':'blueCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_eu['residue_source'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_eu['residue_target'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})

view.addCylinder({
    "start":{'resi': data_eu['residue_source'], 'atom': 'CA'},
    "end":{'resi': data_eu['residue_target'], 'atom': 'CA'},
    "radius":0.1,
    "fromCap":1,
    "toCap":1,
    "color":"red",
    'dashed': True,
    'dashLength': 0.5,
    'gapLength': 0.8 
})

view.rotate(90,'y',1);
view.zoom(0.055)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3D structure with Maximum Angular Separation Distance

In [36]:
data_as = get_data(seqs_with_max_distance, 'angular_separation', dataset_path, pdbs_path)

# display result
display(Markdown(f"**Sequence:** {data_as['sequence']}"))
display(Markdown(f"**Sequence Length:** {data_as['sequence_length']}"))
display(Markdown(f"**Euclidean Distance:** {data_as['euclidean_value']}"))
display(Markdown(f"**Angular Separation Distance:** {data_as['angular_separation_value']}"))

# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_as['pdb_file'], 'r').read(),'pdb')

view.setStyle({'cartoon': {'color':'spectrum', 'thickness':0.4, 'style':'rectangle'}})

view.rotate(90,'y',1);
view.zoom(0.055)

**Sequence:** LQQLLFIHARIGRRRRRRRR

**Sequence Length:** 20

**Euclidean Distance:** 3.698486895512348

**Angular Separation Distance:** 0.999999084343182

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [39]:
# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_as['pdb_file'], 'r').read(),'pdb')

view.addStyle({'and':[{'model':0}]}, {'cartoon': {'opacity':0.8,'color':'white'}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'stick': {'radius': 0.2, 'color':'grey', 'dashedBonds': False, 'singleBonds': False}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'sphere': {'radius': 0.3, 'color':'grey'}})
view.addStyle({'and':[{'model':0},{'atom':'CA'}]}, {'sphere': {'radius': 0.6, 'colorscheme':'blueCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_as['residue_source'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_as['residue_target'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})

view.addCylinder({
    "start":{'resi': data_as['residue_source'], 'atom': 'CA'},
    "end":{'resi': data_as['residue_target'], 'atom': 'CA'},
    "radius":0.1,
    "fromCap":1,
    "toCap":1,
    "color":"red",
    'dashed': True,
    'dashLength': 0.5,
    'gapLength': 0.8 
})


# Scale the direction vectors
residue_source_dir = scalar_vector(data_as['residue_source_dir'], 50)

view.addArrow({
    "start": {'resi':data_as['residue_source'], 'atom':'CA'},  
    "end": {'x': residue_source_dir[0], 'y': residue_source_dir[1], 'z': residue_source_dir[2]},
    "radius": 0.2,
    "color": "yellow"    
})

# Scale the direction vectors
residue_target_dir = scalar_vector(data_as['residue_target_dir'], 30)
 
view.addArrow({
    "start": {'resi':data_as['residue_target'], 'atom':'CA'}, 
    "end": {'x': residue_target_dir[0], 'y': residue_target_dir[1], 'z': residue_target_dir[2]},
    "radius": 0.2,
    "color": "yellow"
})

view.rotate(90,'y');
#view.show()
view.zoom(0.1)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [40]:
# display result
# ALWKNMLKGIGKLAGKAALGAVKKLVGAES (starPep_00022)
#sequence = 'ALWKNMLKGIGKLAGKAALGAVKKLVGAES'
#pdb_file = pdbs_path + 'starPep_00022.pdb'

sequence = 'AGECVQGRCPSGMCCSQFGYCGRGPKYCGR'
pdb_file = pdbs_path + 'starPep_00902.pdb'

display(Markdown(f"**Sequence:** {sequence}"))

# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(pdb_file, 'r').read(),'pdb')

view.setStyle({'cartoon': {'color':'spectrum', 'thickness':0.4, 'style':'rectangle'}})

view.rotate(90,'y',1);
view.zoom(0.055)

**Sequence:** AGECVQGRCPSGMCCSQFGYCGRGPKYCGR

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
view.png()